First we want to test with a static, hardcoded failing Pauli string and circuit

Generate the linked list of all operations that take place in the inverse circuit

In [1]:
class LinkedListNode:
    def __init__(self, name="Initial", depth=0, count=0,  next=None, prev=None):
        self.value = name
        self.depth = depth
        self.next = next
        self.prev = prev
        self.count = count

class LinkedList:
    def __init__(self):
        self.head = None

    def append(self, name, depth):
        new_node = LinkedListNode(name, depth)
        if not self.head:
            self.head = new_node
            return
        last_node = self.head
        while last_node.next:
            last_node = last_node.next
        last_node.next = new_node
        new_node.prev = last_node

    def mark(self, depth):
        current_node = self.head
        while current_node:
            if current_node.depth == depth:
                current_node.count += 1
                return
            current_node = current_node.next

Define our hardcoded Pauli string and CUT

In [2]:
import qiskit.qasm2 as q2
from qiskit import QuantumCircuit

def load_program(name,path):
    qc = QuantumCircuit.from_qasm_file("{}/{}".format(path,name))
    qc.remove_final_measurements()
    if len(qc.clbits) > 0:
        for i in range(len(qc.clbits)):
            qc.measure(i, i)
    else:
        qc.measure_all()
    qc.remove_final_measurements()
    return qc.copy()

static_pauli_string = "IIIIZ"
circuit_under_test = load_program("qnn_indep_qiskit_5.qasm", "benchmarkFilteration/benchmark2/")


Create a list of all instructions within the circuit

In [3]:
circuit_copy = circuit_under_test.copy().inverse()  # Invert to track backward evolution of the operator

operation_list = LinkedList()
depth = 0

operation_list.append("Initial", depth)
for instruction in circuit_copy.data:
    depth += 1
    print(instruction.name, depth)
    operation_list.append(instruction.name, depth)
    

ry 1
ry 2
ry 3
ry 4
ry 5
cx 6
cx 7
cx 8
cx 9
ry 10
ry 11
cx 12
p 13
cx 14
ry 15
cx 16
p 17
cx 18
ry 19
cx 20
p 21
cx 22
ry 23
cx 24
p 25
cx 26
u2 27
cx 28
p 29
cx 30
cx 31
p 32
cx 33
cx 34
p 35
cx 36
u2 37
cx 38
p 39
cx 40
cx 41
p 42
cx 43
cx 44
p 45
cx 46
u2 47
cx 48
p 49
cx 50
cx 51
p 52
cx 53
u2 54
cx 55
p 56
cx 57
u2 58
cx 59
p 60
cx 61
u2 62
cx 63
p 64
cx 65
cx 66
p 67
cx 68
cx 69
p 70
cx 71
u2 72
cx 73
p 74
cx 75
cx 76
p 77
cx 78
u2 79
cx 80
p 81
cx 82
u2 83
u2 84


Observe evolution of Pauli string over CUT with Qiskit

In [4]:
from qiskit.quantum_info import SparsePauliOp, Operator
import numpy as np
from qiskit.quantum_info import Pauli, SparsePauliOp

def evolve_pauli_exact(pauli_label, unitary):
    """Return exact Pauli expansion after conjugation"""
    op = SparsePauliOp(pauli_label).to_operator()
    evolved = unitary.adjoint().compose(op).compose(unitary)
    sp = SparsePauliOp.from_operator(evolved).simplify()

    return {
        p.to_label(): c for p, c in zip(sp.paulis, sp.coeffs)
    }

# 1. Initialize a general quantum circuit with non-Clifford elements
circuit = circuit_under_test.copy().inverse()  # Invert to track backward evolution of the operator

# 2. Define the initial Pauli operator to track
# Qiskit layout is right-to-left: 'IX' means X acts on Qubit 0, I acts on Qubit 1
initial_pauli = SparsePauliOp(static_pauli_string)  
prev_pauli = initial_pauli
depth = 1
num_qubits = circuit.num_qubits

# This dictionary will store our automated history mapping
# Format: { Step_Index: { "gate_type": gate_name, "pauli_strings": { "Pauli_String": Coefficient } } }
evolution_summary_map = {}
ancestry_map = {}  # NEW: step -> {pauli_out: [(pauli_in, contribution), ...]}
transition_graph = {}  # step -> list of (from, to, coeff)

# Log the initial state (Step 0) before any gates apply
evolution_summary_map[0] = {
    "gate_type": "Initial",
    "pauli_strings": {
        pauli.to_label(): np.round(coeff, 4) 
        for pauli, coeff in zip(initial_pauli.paulis, initial_pauli.coeffs)
    }
}

current_layer = {
    pauli.to_label(): coeff
    for pauli, coeff in zip(initial_pauli.paulis, initial_pauli.coeffs)
}

for step_idx, (instruction, qargs, cargs) in enumerate(circuit.data, start=1):

    gate_type = instruction.name
    gate_op = Operator(instruction)

    q_indices = [qarg._index for qarg in qargs]
    full_gate_op = Operator.from_label("I" * num_qubits).compose(gate_op, q_indices)

    new_layer = {}
    edges = []

    for pauli_in, coeff_in in current_layer.items():

        evolved_dict = evolve_pauli_exact(pauli_in, full_gate_op)

        for pauli_out, coeff_out in evolved_dict.items():

            total_coeff = coeff_in * coeff_out

            # accumulate node weights
            new_layer[pauli_out] = new_layer.get(pauli_out, 0) + total_coeff

            # store exact edge
            edges.append({
                "from": pauli_in,
                "to": pauli_out,
                "weight": total_coeff
            })

    # simplify numerical noise (optional but recommended)
    current_layer = {
        k: v for k, v in new_layer.items() if abs(v) > 1e-12
    }

    transition_graph[step_idx] = {
        "gate_type": gate_type,
        "edges": edges
    }

# 4. Print the automated summary map neatly
print("==================================================")
print("   AUTOMATED HEISENBERG EVOLUTION SUMMARY MAP")
print("==================================================")
print("*(Note: Qiskit strings read from right-to-left)*\n")

for step_idx in sorted(transition_graph.keys()):
    step_info = transition_graph[step_idx]
    print(f"Step {step_idx}: Gate = {step_info['gate_type']}")
    print("  Transitions:")
    for edge in step_info["edges"]:
        print(f"    {edge['from']}  --({edge['weight']:.4f})-->  {edge['to']}")
    print("--------------------------------------------------")


C:\Users\Noste\AppData\Local\Temp\ipykernel_500\3879665955.py:45: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for step_idx, (instruction, qargs, cargs) in enumerate(circuit.data, start=1):


   AUTOMATED HEISENBERG EVOLUTION SUMMARY MAP
*(Note: Qiskit strings read from right-to-left)*

Step 1: Gate = ry
  Transitions:
    IIIIZ  --(1.0000+0.0000j)-->  IIIIZ
--------------------------------------------------
Step 2: Gate = ry
  Transitions:
    IIIIZ  --(1.0000+0.0000j)-->  IIIIZ
--------------------------------------------------
Step 3: Gate = ry
  Transitions:
    IIIIZ  --(1.0000+0.0000j)-->  IIIIZ
--------------------------------------------------
Step 4: Gate = ry
  Transitions:
    IIIIZ  --(1.0000+0.0000j)-->  IIIIZ
--------------------------------------------------
Step 5: Gate = ry
  Transitions:
    IIIIZ  --(-0.2665+0.0000j)-->  IIIIX
    IIIIZ  --(0.9638+0.0000j)-->  IIIIZ
--------------------------------------------------
Step 6: Gate = cx
  Transitions:
    IIIIX  --(-0.2665+0.0000j)-->  IIIXX
    IIIIZ  --(0.9638+0.0000j)-->  IIIIZ
--------------------------------------------------
Step 7: Gate = cx
  Transitions:
    IIIXX  --(-0.2665+0.0000j)-->  IIXXX
    

In [12]:
print(transition_graph[1])
print(operation_list.head.next.value)
checked_gate = operation_list.head.next
idx = 1

#For each gate in the circuit
while checked_gate:
    #For each transition from the gate
    for edge in transition_graph[idx]["edges"]:
        if edge["from"] != edge["to"]:
            checked_gate.count += edge["weight"].real**2
    idx += 1
    checked_gate = checked_gate.next

print_gate = operation_list.head
while print_gate:
    print(print_gate.value, print_gate.count)
    print_gate = print_gate.next

{'gate_type': 'ry', 'edges': [{'from': 'IIIIZ', 'to': 'IIIIZ', 'weight': np.complex128(1+0j)}]}
ry
Initial 0
ry 0
ry 0
ry 0
ry 0
ry 0.07102192450936327
cx 0.07102192450936327
cx 0.07102192450936327
cx 0.07102192450936327
cx 0.07102192450936327
ry 0.001589700526760494
ry 0.03952170238327799
cx 0.0323848457339528
p 0.0024448000623421653
cx 0.0347858309095699
ry 0.02317407826996159
cx 0.049135689308076894
p 0.0014744031732689662
cx 0.050446171393867895
ry 0.027525588950826786
cx 0.04550952789779516
p 0.00173270935881679
cx 0.04700296224007422
ry 0.7077095072891983
cx 0.682901672291268
p 0.04393744295544425
cx 0.6854973887031653
u2 0.11034389770751114
cx 0.05997443646589225
p 0.0022933090699250435
cx 0.06061137717451417
cx 0.05758350437526838
p 0.0022627677831017243
cx 0.05830936636038475
cx 0.7035862753391311
p 0.04295822959683922
cx 0.7048478854509378
u2 0.10581577968615423
cx 0.11161102233538316
p 0.00596664579596491
cx 0.11347270585657382
cx 0.052379960682705694
p 0.0020343275359699206